# 9.12 Instruction tuning

Instruction tuning turns a pretrained LM into a helpful interface by training on prompt-response pairs.

This notebook implements response-only SFT loss and an instruction-following feature model. It shows why prompt masking, mixture weights, format tokens, and supervised-token counts matter.

Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

SEED = 912
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)


def sigmoid(values):
    return 1.0 / (1.0 + np.exp(-values))


def build_instruction_ladder():
    rng = np.random.default_rng(SEED)
    ladders = []
    ladders.append({"name": "D1 one prompt", "probs": np.array([0.8, 0.5, 0.25]), "prompt_tokens": 5, "response_tokens": 3, "format_gain": 1.5, "labels": np.array([1]), "scores": np.array([0.8])})
    ladders.append({"name": "D2 few-shot instructions", "prompt_tokens": 6, "response_tokens": 4, "scores": np.array([0.7, 0.8, 0.6, 0.9, 0.75]), "labels": np.ones(5)})
    ladders.append({"name": "D3 rare behavior", "prompt_tokens": 9, "response_tokens": 5, "scores": np.array([0.85, 0.65, 0.40, 0.90, 0.55, 0.35]), "labels": np.array([1, 1, 1, 1, 0, 0])})
    ladders.append({"name": "D4 instruction corpus", "prompt_tokens": 12, "response_tokens": 8, "scores": np.array([0.82, 0.77, 0.74, 0.66, 0.71, 0.69, 0.73, 0.80]), "labels": np.ones(8)})
    scores_d5 = np.clip(rng.normal(loc=0.68, scale=0.18, size=40), 0.05, 0.98)
    scores_d5[::6] -= 0.35
    scores_d5 = np.clip(scores_d5, 0.05, 0.98)
    labels_d5 = np.ones_like(scores_d5)
    ladders.append({"name": "D5 format shift", "prompt_tokens": 40, "response_tokens": 20, "scores": scores_d5, "labels": labels_d5})
    return ladders


def instruction_accuracy(scores, labels):
    predictions = (scores >= 0.5).astype(int)
    return float(np.mean(predictions == labels))


## The concept, built once (D1)

The response-only SFT objective is $$\mathcal{L}_{\mathrm{SFT}}=-\sum_{t\in R}\log p_\theta(y_t\mid x,y_{\lt t}).$$
The lesson asserts response probabilities $0.8,0.5,0.25$ give loss $2.303$, a 5-token prompt plus 3-token response means $37.5\%$ supervised positions, $0.7\cdot1.0+0.3\cdot2.0=1.300$, $e^{1.5}=4.482$, and $10{,}000\times200=2{,}000{,}000$ supervised tokens.

In [ ]:

def sft_loss(token_probs, prompt_tokens, response_tokens, response_only=True):
    token_probs = np.asarray(token_probs, dtype=float)
    response_loss = float(np.sum(-np.log(token_probs + 1e-12)))
    total_tokens = prompt_tokens + response_tokens
    supervised_fraction = response_tokens / total_tokens if response_only else 1.0
    return response_loss, supervised_fraction

loss_demo, supervised_fraction_demo = sft_loss(np.array([0.8, 0.5, 0.25]), prompt_tokens=5, response_tokens=3)
mixture_loss = 0.7 * 1.0 + 0.3 * 2.0
odds_gain = math.exp(1.5)
supervised_tokens = 10_000 * 200
assert round(loss_demo, 3) == 2.303
assert round(supervised_fraction_demo * 100, 1) == 37.5
assert round(mixture_loss, 3) == 1.300
assert round(odds_gain, 3) == 4.482
assert supervised_tokens == 2_000_000
print(loss_demo, supervised_fraction_demo, mixture_loss, odds_gain, supervised_tokens)


The same mask distinguishes answering from echoing. We treat format control as a logit gain that changes odds by an exponential factor.

In [ ]:

base_logit = 0.0
format_gain = 1.5
base_prob = sigmoid(base_logit)
formatted_prob = sigmoid(base_logit + format_gain)
print("base format probability", base_prob)
print("formatted probability", formatted_prob)
print("odds multiplier", math.exp(format_gain))


## The dataset ladder

The ladder starts with one prompt-response pair and grows toward a D5 format shift where prompt echoing becomes tempting.

In [ ]:

ladder = build_instruction_ladder()
for rung in ladder:
    print(rung["name"], "prompt", rung["prompt_tokens"], "response", rung["response_tokens"], "examples", len(rung["scores"]))
    print("score sample", rung["scores"][:5])


## Run the same instruction-tuning metric across D1-D5

In [ ]:

results = []
for rung in ladder:
    response_loss, supervised_fraction = sft_loss(np.clip(rung["scores"], 1e-6, 1.0), rung["prompt_tokens"], rung["response_tokens"])
    accuracy = instruction_accuracy(rung["scores"], rung["labels"])
    perplexity = math.exp(response_loss / max(len(rung["scores"]), 1))
    results.append({"name": rung["name"], "loss": response_loss, "fraction": supervised_fraction, "metric": accuracy, "perplexity": perplexity, "scores": rung["scores"]})

print("rung | instruction_accuracy | response_perplexity | supervised_fraction")
for result in results:
    print(f"{result['name']} | {result['metric']:.3f} | {result['perplexity']:.3f} | {result['fraction']:.3f}")


## Results visualization

Small multiples show score distributions. The summary curve tracks instruction-following accuracy.

In [ ]:

fig, axes = plt.subplots(1, len(results), figsize=(16, 3))
for ax, result in zip(axes, results):
    ax.hist(result["scores"], bins=np.linspace(0.0, 1.0, 8), color="steelblue")
    ax.axvline(0.5, color="black", linestyle="--")
    ax.set_title(result["name"].split()[0])
    ax.set_xlabel("follow score")
    ax.set_ylabel("count")
plt.tight_layout()

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(range(1, len(results) + 1), [item["metric"] for item in results], marker="o")
ax.set_xticks(range(1, len(results) + 1))
ax.set_xticklabels(["D1", "D2", "D3", "D4", "D5"])
ax.set_ylabel("instruction-following accuracy")
ax.set_title("SFT quality across ladder")
plt.show()


## Pitfall on the hardest rung

Pitfall: putting loss on prompt tokens trains echoing. On D5, compare a wrong all-token mask with the response-only mask.

In [ ]:

d5 = ladder[-1]
response_loss, response_fraction = sft_loss(np.clip(d5["scores"], 1e-6, 1.0), d5["prompt_tokens"], d5["response_tokens"], response_only=True)
wrong_loss, wrong_fraction = sft_loss(np.clip(d5["scores"], 1e-6, 1.0), d5["prompt_tokens"], d5["response_tokens"], response_only=False)
echo_rate_wrong = d5["prompt_tokens"] / (d5["prompt_tokens"] + d5["response_tokens"])
echo_rate_fixed = 0.0
print("wrong supervised fraction", wrong_fraction, "echo pressure", echo_rate_wrong)
print("fixed supervised fraction", response_fraction, "echo pressure", echo_rate_fixed)
print("response loss", response_loss)


## Evaluate it + Practice

- Metric: instruction-following accuracy, with response perplexity as a loss diagnostic.
- No-skill baseline: prompt echo baseline repeats the instruction and may look fluent but fails labels.
- Cheap sanity check: response-only mask must supervise exactly $3/8=37.5\%$ in the worked example.
- Ablation: train with all-token loss and observe D5 echo pressure rise.
- Failure signals: format-sensitive accuracy drops, rare behaviors drowned by mixture weights, or prompt-copy outputs.

Practice prompts:


**Practice.** Change chat/code mixture weights and recompute combined loss.

**Practice.** Add a format-token gain of 0.5 instead of 1.5 and compare odds.

**Practice.** Make D5 prompt length longer and explain the echo pressure.